# PRE_05 — Anonimización

`data/raw.csv` tiene 600 clientes con identificadores directos (nombre, cédula,
correo, tarjeta de fidelidad) y cuasi-identificadores (edad, ciudad, ocupación).

Borrar los identificadores directos **no alcanza**: la combinación
(edad, ciudad, ocupación) es única en 556 de los 600 registros. `data/auxiliary.csv`
es justo esa información con nombre y apellido — es decir, el ataque de
re-identificación es real y directo.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))
import pandas as pd
from PRE_05_anonimizacion.src.anonymize import (
    INPUT_FILE, AUXILIARY_FILE, QUASI_IDENTIFIERS,
    anonymize, generalize, drop_direct_identifiers,
    k_anonymity, reidentification_risk, main,
)

raw = pd.read_csv(INPUT_FILE)
raw.head()

## El riesgo antes de anonimizar

Cuántos registros son únicos por sus cuasi-identificadores.

In [ ]:
sizes = raw.groupby(["age", "city", "occupation"]).size()
print("combinaciones:", len(sizes))
print("únicas (k=1):", (sizes == 1).sum(), "de", len(raw), "registros")

auxiliary = pd.read_csv(AUXILIARY_FILE)
auxiliary.head()

## Estrategia

1. Eliminar los identificadores directos.
2. Generalizar los cuasi-identificadores: edad a rangos, ciudad a región, ocupación a sector.
3. Suprimir los grupos que aun así queden con menos de `k` personas.

El resultado cumple k-anonimato: cada registro es indistinguible de al menos otros k-1.

In [ ]:
anonymized = main()
anonymized.head(10)

## Elección de k y del ancho de los rangos de edad

Hay un intercambio directo entre privacidad y utilidad. Con bandas de 10 años
y k=5 se pierde el 44% de los registros; con tres bandas la pérdida baja al 26%
sin bajar el k.

In [ ]:
import PRE_05_anonimizacion.src.anonymize as an

configs = {
    "5 bandas": ([0,29,39,49,59,200], ["20-29","30-39","40-49","50-59","60+"]),
    "3 bandas (elegida)": ([0,34,49,200], ["20-34","35-49","50+"]),
    "2 bandas": ([0,44,200], ["20-44","45+"]),
}
original_bins, original_labels = an.AGE_BINS, an.AGE_LABELS
filas = []
for nombre, (bins, labels) in configs.items():
    an.AGE_BINS, an.AGE_LABELS = bins, labels
    for k in (3, 5):
        df = an.anonymize(k=k)
        filas.append({
            "config": nombre, "k objetivo": k,
            "filas": len(df), "conservado": f"{len(df)/len(raw):.1%}",
            "k real": an.k_anonymity(df),
            "identificadas": an.reidentification_risk(df)["identificadas"],
        })
an.AGE_BINS, an.AGE_LABELS = original_bins, original_labels
pd.DataFrame(filas)

## Verificación del ataque

Se generalizan los 20 nombres de `auxiliary.csv` igual que la tabla publicada
y se mira a cuántos registros corresponde cada persona. Ninguna debe quedar sola.

In [ ]:
reidentification_risk(anonymized)

`identificadas: 0` — ninguna de las 20 personas del archivo auxiliar queda
identificada de forma única. `minimo_candidatos: 5` es el peor caso: la persona
más expuesta se confunde con otras 4.